# 504 — Phase-1 Hardening Smoke Tests

Lightweight assertions that validate the consistency of the phase-1
auth / role / trace-context implementation after the cleanup in Prompt 4.

**No live Neo4j connection required.** All checks use static imports,
object construction, and dict lookups.

In [1]:
import sys
sys.path.insert(0, "..")

from src.app.auth import (
    AuthenticatedUser,
    authenticate,
    dev_bypass_user,
    get_mock_users,
    is_dev_bypass_enabled,
)
from src.app.policy import (
    ROLE_JR_RISK_ANALYST,
    ROLE_SR_RISK_ANALYST,
    ADDRESS_RISK_TOOLS,
    INDUSTRY_RISK_TOOLS,
    ALL_MCP_TOOLS,
    JR_ALLOWED_MCP_TOOLS,
    get_policy,
    get_policy_for_user,
)
from src.domain.models import InvestigationTrace, UserContext

print("Imports OK")

Imports OK


## 1. Auth provider constants are stable

`AuthenticatedUser.auth_provider` defaults to `"mock"`.  
`dev_bypass_user()` sets `"dev_bypass"`.  
These are the two valid phase-1 values; `"kong"` is reserved for phase 2.

In [2]:
# Default auth_provider is "mock"
mock_user = authenticate("sr_analyst", "demo123")
assert mock_user is not None, "sr_analyst authentication failed"
assert mock_user.auth_provider == "mock", (
    f"Expected auth_provider='mock', got '{mock_user.auth_provider}'"
)

# dev_bypass_user sets auth_provider to "dev_bypass"
bypass = dev_bypass_user()
assert bypass.auth_provider == "dev_bypass", (
    f"Expected auth_provider='dev_bypass', got '{bypass.auth_provider}'"
)

# AuthenticatedUser dataclass default field
manual_user = AuthenticatedUser(user_id="test", role="sr_risk_analyst")
assert manual_user.auth_provider == "mock", (
    f"Default auth_provider should be 'mock', got '{manual_user.auth_provider}'"
)

print("PASS — auth_provider constants: mock, dev_bypass")

PASS — auth_provider constants: mock, dev_bypass


## 2. Role identifiers are consistent across auth and policy

The stable role strings in `policy.py` (`ROLE_JR_RISK_ANALYST`, `ROLE_SR_RISK_ANALYST`)
must match the roles assigned in `auth._MOCK_USERS`.

In [3]:
mock_users = get_mock_users()

# Verify every mock user role is a known policy role identifier
known_roles = {ROLE_JR_RISK_ANALYST, ROLE_SR_RISK_ANALYST}
for username, entry in mock_users.items():
    role = entry["role"]
    assert role in known_roles, (
        f"Mock user '{username}' has role '{role}' which is not a known policy role. "
        f"Known: {known_roles}"
    )

# Spot-check the specific assignments
assert mock_users["jr_analyst"]["role"] == ROLE_JR_RISK_ANALYST
assert mock_users["sr_analyst"]["role"] == ROLE_SR_RISK_ANALYST

print(f"PASS — all {len(mock_users)} mock users map to known policy roles")

PASS — all 2 mock users map to known policy roles


## 3. Jr analyst policy capabilities

In [4]:
jr = get_policy(ROLE_JR_RISK_ANALYST)

assert jr.can_investigate is True,        "Jr should be able to investigate"
assert jr.can_replay is False,             "Jr should NOT be able to replay"
assert jr.can_view_tech_evidence is True,  "Jr should be able to view tech evidence"

# Restricted tools
assert not jr.can_invoke_mcp_tool("address_risk_check"),   "Jr must not invoke address_risk_check"
assert not jr.can_invoke_mcp_tool("industry_context_check"), "Jr must not invoke industry_context_check"
assert not jr.can_invoke_mcp_tool("summarize_risk_for_company"), "Jr must not invoke summarize_risk_for_company"

# Allowed tools
for tool in ["entity_lookup", "company_profile", "expand_ownership",
             "shared_address_check", "sic_context",
             "ownership_complexity_check", "control_signal_check",
             "retrieve_trace", "find_traces_by_entity", "list_recent_traces",
             "resolve_entity", "validate_plan", "evaluate_stop_conditions"]:
    assert jr.can_invoke_mcp_tool(tool), f"Jr should be allowed to invoke '{tool}'"

print("PASS — Jr policy capabilities correct")

PASS — Jr policy capabilities correct


## 4. Sr analyst policy capabilities

In [5]:
sr = get_policy(ROLE_SR_RISK_ANALYST)

assert sr.can_investigate is True
assert sr.can_replay is True
assert sr.can_view_tech_evidence is True

# Sr can invoke everything in ALL_MCP_TOOLS
for tool in ALL_MCP_TOOLS:
    assert sr.can_invoke_mcp_tool(tool), f"Sr should be allowed to invoke '{tool}'"

# Sr allowed_mcp_tools == ALL_MCP_TOOLS
assert sr.allowed_mcp_tools == ALL_MCP_TOOLS, (
    f"Sr allowed_mcp_tools should equal ALL_MCP_TOOLS. "
    f"Diff: {sr.allowed_mcp_tools.symmetric_difference(ALL_MCP_TOOLS)}"
)

print("PASS — Sr policy has full access")

PASS — Sr policy has full access


## 5. Unknown role → deny-all fallback

The policy registry must fail closed for unrecognised role strings.

In [6]:
deny = get_policy("unknown_role")

assert deny.can_investigate is False
assert deny.can_replay is False
assert deny.can_view_tech_evidence is False
assert len(deny.allowed_mcp_tools) == 0, (
    f"Deny-all policy should have no allowed tools, got: {deny.allowed_mcp_tools}"
)

print("PASS — unknown role produces deny-all policy")

PASS — unknown role produces deny-all policy


## 6. Trace identity fields are present and round-trip correctly

Confirms all 5 phase-1 identity fields exist on `InvestigationTrace`
with correct defaults.

In [7]:
import uuid

session_id = str(uuid.uuid4())
trace = InvestigationTrace(
    request_id="req-001",
    entity_name="Test Corp",
    user_id="sr_analyst",
    mode="investigate",
    user_role="sr_risk_analyst",
    auth_provider="mock",
    session_id=session_id,
    gateway_mode="local",
)

assert trace.user_id == "sr_analyst"
assert trace.user_role == "sr_risk_analyst"
assert trace.auth_provider == "mock"
assert trace.session_id == session_id
assert trace.gateway_mode == "local"
assert trace.mode == "investigate"

# Default values for identity fields when not supplied
minimal = InvestigationTrace(request_id="req-002")
assert minimal.user_role == ""
assert minimal.auth_provider == ""
assert minimal.session_id == ""
assert minimal.gateway_mode == ""

print("PASS — InvestigationTrace identity fields round-trip correctly")

PASS — InvestigationTrace identity fields round-trip correctly


## 7. Backward-compatible old-trace loading

Old traces loaded from Neo4j will be missing the new identity fields.
`dict.get()` with a default must return `""` (not `None` or `KeyError`).

In [8]:
# Simulates a trace dict loaded from an old Neo4j node missing the new fields
old_trace_dict = {
    "trace_id": "old-trace-001",
    "query": "ACME Ltd",
    "user_id": "legacy_user",
    "mode": "investigate",
    "started_at": "2024-01-01T00:00:00+00:00",
    "ended_at": None,
    "final_summary": None,
    # user_role, auth_provider, session_id, gateway_mode intentionally absent
}

assert (old_trace_dict.get("user_role") or "") == ""
assert (old_trace_dict.get("auth_provider") or "") == ""
assert (old_trace_dict.get("session_id") or "") == ""
assert (old_trace_dict.get("gateway_mode") or "") == ""

# Confirm the existing fields still work
assert old_trace_dict.get("user_id") == "legacy_user"
assert old_trace_dict.get("mode") == "investigate"

print("PASS — old-trace backward compatibility: missing fields degrade gracefully")

PASS — old-trace backward compatibility: missing fields degrade gracefully


## 8. Gateway mode naming is consistent

`UserContext.metadata["gateway_mode"]` must align with `InvestigationTrace.gateway_mode`.
This checks the key name convention used in `layout.py` when building `UserContext`.

In [9]:
import dataclasses

# The metadata key used in layout.py
LAYOUT_GATEWAY_KEY = "gateway_mode"

# The field name on InvestigationTrace
trace_fields = {f.name for f in dataclasses.fields(InvestigationTrace)}
assert LAYOUT_GATEWAY_KEY in trace_fields, (
    f"InvestigationTrace is missing field '{LAYOUT_GATEWAY_KEY}'. "
    f"Fields: {trace_fields}"
)

# Confirm UserContext can carry it
ctx = UserContext(
    user_id="sr_analyst",
    session_id=str(uuid.uuid4()),
    metadata={"role": "sr_risk_analyst", "auth_provider": "mock", "gateway_mode": "local"},
)
assert ctx.metadata.get("gateway_mode") == "local"

print("PASS — 'gateway_mode' key is consistent in UserContext.metadata and InvestigationTrace")

PASS — 'gateway_mode' key is consistent in UserContext.metadata and InvestigationTrace


## 9. Mock user registry ↔ policy alignment

Every role in the mock user registry must map to a non-deny-all policy.

In [10]:
deny_role = "__deny_all__"
mock_users = get_mock_users()

for username, entry in mock_users.items():
    role = entry["role"]
    policy = get_policy(role)
    assert policy.role != deny_role, (
        f"Mock user '{username}' with role '{role}' resolved to deny-all policy. "
        "Add this role to the policy registry in src/app/policy.py."
    )
    assert policy.can_investigate, (
        f"Mock user '{username}' with role '{role}' has can_investigate=False — unexpected."
    )

# Also verify get_policy_for_user works with AuthenticatedUser
user = authenticate("sr_analyst", "demo123")
assert user is not None
policy_via_user = get_policy_for_user(user)
assert policy_via_user.role == ROLE_SR_RISK_ANALYST

print(f"PASS — all {len(mock_users)} mock users have a valid, non-deny-all policy")

PASS — all 2 mock users have a valid, non-deny-all policy


## 10. Compile check

In [11]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "compileall", "-q", "../src", "../app.py"],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
assert result.returncode == 0, "compileall failed — syntax error in src/ or app.py"
print("PASS — compileall clean")

PASS — compileall clean


---

## Manual UI checks still required

These cannot be automated in this notebook. Run `streamlit run app.py` and verify:

1. **Login gate** — appears on fresh load; invalid credentials show an error message.
2. **Jr analyst flow** — login as `jr_analyst`/`demo123`:
   - Only the **Investigate** tab is visible (no Replay / Audit).
   - Investigation runs; `address_risk_check` and `industry_context_check` steps are skipped with a policy warning.
3. **Sr analyst flow** — login as `sr_analyst`/`demo123`:
   - Both **Investigate** and **Replay / Audit** tabs are visible.
   - Full investigation runs including address-risk and industry-risk steps.
   - Load the just-created trace ID in the Replay tab; the **Session Identity** expander (inside "How this was assessed") shows: user, role, auth_provider, session, gateway_mode.
4. **Dev bypass** — set `DEV_BYPASS_AUTH=true` in `.env`, reload; a bypass button appears on the login screen and logs in as `sr_risk_analyst`.
5. **Sign out / sign in** — sign out and sign back in as a different user; no stale results from the prior session appear in either tab.
6. **Neo4j persistence check** (optional) — run in a Cypher browser:
   ```cypher
   MATCH (t:InvestigationTrace)
   RETURN t.user_id, t.user_role, t.auth_provider, t.session_id, t.gateway_mode
   ORDER BY t.started_at DESC LIMIT 5
   ```
   All five fields should be populated for recently created traces.